Caderno Jupyter para testes de treinamento em ML

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn 

In [ ]:
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
df["Sleep Disorder"] = df["Sleep Disorder"].fillna("None")
# print(df["Sleep Disorder"].value_counts())

In [ ]:
contagem = df["Sleep Disorder"].value_counts()
total = contagem.sum()

plt.figure(figsize=(7,5))
ax = contagem.plot(kind="bar", color=["#66b3ff", "#ff9999", "#99ff99"])

plt.title("Distribuição dos Distúrbios do Sono", fontsize=14)
plt.xlabel("Tipo de Distúrbio")
plt.ylabel("Quantidade de Pessoas")

for i, v in enumerate(contagem):
    percentual = (v / total) * 100
    ax.text(i, v + 1, f"{v} ({percentual:.1f}%)", 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.show()

In [ ]:
# Mantém apenas as linhas com algum distúrbio (descarta 'None')
df = df[df["Sleep Disorder"].notna()]

# Mantém apenas Insomnia e Sleep Apnea
df = df[df["Sleep Disorder"].isin(["Insomnia", "Sleep Apnea"])]

# Coluna alvo (o que queremos prever)
y = df["Sleep Disorder"]

# Colunas que vamos usar para prever
X = df[["Age", "Sleep Duration", "Quality of Sleep",
        "Physical Activity Level", "Stress Level",
        "Heart Rate", "Daily Steps"]]

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Ver quais números correspondem a quais distúrbios
print(dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
from sklearn.ensemble import RandomForestClassifier

modelo = RandomForestClassifier(
    random_state=42,
    max_depth=5,          # limita profundidade das árvores
    min_samples_leaf=5,   # exige mínimo de amostras nas folhas
    n_estimators=200      # mais árvores ajuda a estabilizar
)
modelo.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import classification_report

y_pred = modelo.predict(X_test)

print("Relatório de Classificação:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))



In [ ]:
print("Acurácia treino:", modelo.score(X_train, y_train))
print("Acurácia teste:", modelo.score(X_test, y_test))

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scores = cross_val_score(modelo, X, y_encoded, cv=rkf, scoring='accuracy')
print(f"Acurácia média: {scores.mean():.3f} (+/- {scores.std():.3f})")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

modelos = {
    "Random Forest": modelo,
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

for nome, m in modelos.items():
    scores = cross_val_score(m, X, y_encoded, cv=rkf, scoring='accuracy')
    print(f"{nome}: {scores.mean():.3f} (+/- {scores.std():.3f})")